# Exercise 4: Effect of Top-K Retrieval Count

Vary the number of retrieved chunks (k) and observe how it affects answer quality.

**Test values:** k = 1, 3, 5, 10, 20


In [1]:
# Install required packages
try:
    ip = get_ipython()
    ip.run_line_magic('pip', 'install -q torch transformers sentence-transformers faiss-cpu pymupdf accelerate')
except NameError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'torch', 'transformers', 'sentence-transformers', 'faiss-cpu', 'pymupdf', 'accelerate'])


In [2]:
import os, sys
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
from typing import List, Tuple
from pathlib import Path

def detect_environment():
    try:
        import google.colab
        return 'colab'
    except ImportError:
        return 'local'

def get_device():
    if torch.cuda.is_available():
        device, dtype = 'cuda', torch.float16
        print(f"✓ CUDA: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
    elif torch.backends.mps.is_available():
        device, dtype = 'mps', torch.float32
        print("✓ Apple Silicon MPS")
    else:
        device, dtype = 'cpu', torch.float32
        print("⚠ CPU only")
    return device, dtype

ENVIRONMENT = detect_environment()
DEVICE, DTYPE = get_device()
print(f"Environment: {ENVIRONMENT.upper()} | Device: {DEVICE} | Dtype: {DTYPE}")


✓ CUDA: Tesla T4 (15.6 GB)
Environment: COLAB | Device: cuda | Dtype: torch.float16


In [3]:
from pathlib import Path

CORPUS_SUBFOLDER = "ModelTService"  # <- change if needed
DRIVE_BASE = '/content/drive/MyDrive/Colab_Projects/Week05-RAG/Corpora'

if ENVIRONMENT == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    DOC_FOLDER = f"{DRIVE_BASE}/{CORPUS_SUBFOLDER}"
else:
    DOC_FOLDER = str(Path("Corpora") / CORPUS_SUBFOLDER)

print(f"DOC_FOLDER = {DOC_FOLDER}")
assert Path(DOC_FOLDER).exists(), f"Folder not found: {DOC_FOLDER}"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DOC_FOLDER = /content/drive/MyDrive/Colab_Projects/Week05-RAG/Corpora/ModelTService


In [4]:
import fitz

def load_text_file(fp):
    with open(fp, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()

def load_pdf_file(fp):
    doc = fitz.open(fp)
    parts = []
    for i, page in enumerate(doc):
        t = page.get_text()
        if t.strip():
            parts.append(f"\n[Page {i+1}]\n{t}")
    doc.close()
    return "\n".join(parts)

def load_documents(folder):
    docs = []
    for fp in Path(folder).rglob("*"):
        if not fp.is_file(): continue
        if fp.suffix.lower() not in ('.pdf', '.txt', '.md'): continue
        try:
            content = load_pdf_file(str(fp)) if fp.suffix.lower() == '.pdf' else load_text_file(str(fp))
            if content.strip():
                docs.append((fp.name, content))
                print(f"✓ {fp.name} ({len(content):,} chars)")
        except Exception as e:
            print(f"✗ {fp.name}: {e}")
    return docs

documents = load_documents(DOC_FOLDER)
print(f"\nLoaded {len(documents)} documents")


✓ Ford-Model-T-Man-1919.txt (95,574 chars)
✓ ModelT-01-10.txt (18,676 chars)
✓ ModelT-11-20.txt (19,009 chars)
✓ ModelT-21-30.txt (17,050 chars)
✓ ModelT-31-40.txt (12,194 chars)
✓ ModelT-41-50.txt (14,264 chars)
✓ ModelT-51-60.txt (14,168 chars)
✓ ModelT-61-62.txt (201 chars)
✓ Ford-Model-T-Man-1919-ocr.pdf (95,517 chars)
✓ ModelT-01-10-ocr.pdf (18,658 chars)
✓ ModelT-11-20-ocr.pdf (19,003 chars)
✓ ModelT-21-30-ocr.pdf (17,025 chars)
✓ ModelT-31-40-ocr.pdf (12,201 chars)
✓ ModelT-41-50-ocr.pdf (14,270 chars)
✓ ModelT-51-60-ocr.pdf (14,107 chars)
✓ ModelT-61-62-ocr.pdf (204 chars)

Loaded 16 documents


In [5]:
from dataclasses import dataclass

@dataclass
class Chunk:
    text: str
    source_file: str
    chunk_index: int
    start_char: int
    end_char: int

def chunk_text(text, source_file, chunk_size=512, chunk_overlap=128):
    chunks, start, idx = [], 0, 0
    while start < len(text):
        end = start + chunk_size
        if end < len(text):
            pb = text.rfind('\n\n', start + chunk_size // 2, end)
            if pb != -1:
                end = pb + 2
            else:
                sb = text.rfind('. ', start + chunk_size // 2, end)
                if sb != -1:
                    end = sb + 2
        s = text[start:end].strip()
        if s:
            chunks.append(Chunk(s, source_file, idx, start, end))
            idx += 1
        prev_start = start
        start = end - chunk_overlap
        if chunks and start <= chunks[-1].start_char:
            start = end
    return chunks

# Default chunking parameters (override per exercise)
CHUNK_SIZE    = 512
CHUNK_OVERLAP = 128

all_chunks = []
for fname, content in documents:
    dc = chunk_text(content, fname, CHUNK_SIZE, CHUNK_OVERLAP)
    all_chunks.extend(dc)
    print(f"  {fname}: {len(dc)} chunks")
print(f"\nTotal chunks: {len(all_chunks)}")


  Ford-Model-T-Man-1919.txt: 326 chunks
  ModelT-01-10.txt: 64 chunks
  ModelT-11-20.txt: 66 chunks
  ModelT-21-30.txt: 56 chunks
  ModelT-31-40.txt: 44 chunks
  ModelT-41-50.txt: 51 chunks
  ModelT-51-60.txt: 46 chunks
  ModelT-61-62.txt: 1 chunks
  Ford-Model-T-Man-1919-ocr.pdf: 316 chunks
  ModelT-01-10-ocr.pdf: 63 chunks
  ModelT-11-20-ocr.pdf: 61 chunks
  ModelT-21-30-ocr.pdf: 56 chunks
  ModelT-31-40-ocr.pdf: 44 chunks
  ModelT-41-50-ocr.pdf: 47 chunks
  ModelT-51-60-ocr.pdf: 44 chunks
  ModelT-61-62-ocr.pdf: 1 chunks

Total chunks: 1286


In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
print(f"Loading: {EMBEDDING_MODEL} on {DEVICE}")
embed_model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
EMBEDDING_DIM = embed_model.get_sentence_embedding_dimension()
print(f"Embedding dim: {EMBEDDING_DIM}")


Loading: sentence-transformers/all-MiniLM-L6-v2 on cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dim: 384


In [7]:
import faiss, pickle
from pathlib import Path

# Cache config — CACHE_KEY encodes corpus + chunk params
# Change it if you change CORPUS_SUBFOLDER, CHUNK_SIZE, or CHUNK_OVERLAP
CACHE_DIR   = Path("/content/drive/MyDrive/Colab_Projects/Week05-RAG/cache")
CACHE_KEY   = "modelTService_512_128"
CHUNKS_FILE = CACHE_DIR / f"{CACHE_KEY}.chunks.pkl"
EMBEDS_FILE = CACHE_DIR / f"{CACHE_KEY}.embeddings.npy"
INDEX_FILE  = CACHE_DIR / f"{CACHE_KEY}.faiss"
CACHE_DIR.mkdir(exist_ok=True)

def save_cache():
    with open(CHUNKS_FILE, "wb") as f: pickle.dump(all_chunks, f)
    np.save(str(EMBEDS_FILE), chunk_embeddings)
    faiss.write_index(index, str(INDEX_FILE))
    print(f"✓ Cache saved → {CACHE_DIR}/{CACHE_KEY}.*")

def load_cache():
    global all_chunks, chunk_embeddings, index
    with open(CHUNKS_FILE, "rb") as f: all_chunks = pickle.load(f)
    chunk_embeddings = np.load(str(EMBEDS_FILE))
    index = faiss.read_index(str(INDEX_FILE))
    print(f"✓ Cache loaded: {len(all_chunks)} chunks, {index.ntotal} vectors")

if CHUNKS_FILE.exists() and EMBEDS_FILE.exists() and INDEX_FILE.exists():
    load_cache()
else:
    print("No cache found — embedding chunks (first run only, will be cached after)...")
    chunk_embeddings = embed_model.encode(
        [c.text for c in all_chunks], show_progress_bar=True
    ).astype("float32")
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    faiss.normalize_L2(chunk_embeddings)
    index.add(chunk_embeddings)
    print(f"✓ Index built: {index.ntotal} vectors")
    save_cache()

def rebuild_pipeline(chunk_size=512, chunk_overlap=128):
    """Re-chunk, re-embed, rebuild index in-memory (for Ex 7 & 8 experiments)."""
    global all_chunks, chunk_embeddings, index
    all_chunks = []
    for fname, content in documents:
        all_chunks.extend(chunk_text(content, fname, chunk_size, chunk_overlap))
    chunk_embeddings = embed_model.encode(
        [c.text for c in all_chunks], show_progress_bar=True
    ).astype("float32")
    faiss.normalize_L2(chunk_embeddings)
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    index.add(chunk_embeddings)
    print(f"Rebuilt: {len(all_chunks)} chunks | size={chunk_size}, overlap={chunk_overlap}")

def retrieve(query, top_k=5):
    qe = embed_model.encode([query]).astype("float32")
    faiss.normalize_L2(qe)
    scores, idxs = index.search(qe, top_k)
    return [(all_chunks[i], float(s)) for s, i in zip(scores[0], idxs[0]) if i != -1]


✓ Cache loaded: 1286 chunks, 1286 vectors


In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer

LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading LLM: {LLM_MODEL} on {DEVICE}...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

if DEVICE == 'cuda':
    model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, device_map="auto",
                torch_dtype=DTYPE, trust_remote_code=True)
elif DEVICE == 'mps':
    model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=DTYPE,
                trust_remote_code=True).to(DEVICE)
else:
    model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=DTYPE,
                trust_remote_code=True)
print("LLM loaded ✓")


Loading LLM: Qwen/Qwen2.5-1.5B-Instruct on cuda...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

LLM loaded ✓


In [9]:
PROMPT_TEMPLATE = """You are a helpful assistant that answers questions based on the provided context.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS:
- Answer ONLY based on the information in the context above.
- If the context doesn't contain the answer, say so explicitly.
- Quote relevant passages to support your answer.
- Be concise and direct.

ANSWER:"""

def generate_response(prompt, max_new_tokens=512, temperature=0.3):
    inputs = tokenizer(prompt, return_tensors="pt")
    if DEVICE == 'cuda':
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    else:
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             temperature=temperature,
                             do_sample=(temperature > 0),
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def direct_query(question, max_new_tokens=512):
    prompt = f"Answer this question:\n{question}\n\nAnswer:"
    return generate_response(prompt, max_new_tokens)

def rag_query(question, top_k=5, show_context=False, prompt_template=None):
    results = retrieve(question, top_k)
    ctx_parts = [f"[Source: {c.source_file}, Score: {s:.3f}]\n{c.text}" for c, s in results]
    context = "\n\n---\n\n".join(ctx_parts)
    if show_context:
        print("=== RETRIEVED CONTEXT ==="); print(context); print("=" * 40)
    tmpl = prompt_template if prompt_template else PROMPT_TEMPLATE
    return generate_response(tmpl.format(context=context, question=question))


In [10]:
import time

TEST_QUERIES = [
    "How do I adjust the carburetor on a Model T?",
    "What is the correct spark plug gap for a Model T Ford?",
    "How do I fix a slipping transmission band?",
    "What oil should I use in a Model T engine?",
    "How do I adjust the engine timing?",
]

K_VALUES = [1, 3, 5, 10, 20]

results_table = []  # (k, query, latency, answer_len, answer)

for k in K_VALUES:
    print(f"\n{'='*70}")
    print(f"TOP-K = {k}")
    print(f"{'='*70}")
    for q in TEST_QUERIES:
        print(f"\n  Q: {q}")
        t0 = time.time()
        answer = rag_query(q, top_k=k)
        latency = time.time() - t0
        results_table.append((k, q[:40], latency, len(answer), answer[:200]))
        print(f"  [k={k}] ({latency:.1f}s, {len(answer)} chars): {answer[:200]}...")



TOP-K = 1

  Q: How do I adjust the carburetor on a Model T?
  [k=1] (24.8s, 2653 chars): To adjust the carburetor on a Model T, you would need to refer to the specific instructions or diagrams provided with the car's manual. The text mentions "install a new one," which implies that if you...

  Q: What is the correct spark plug gap for a Model T Ford?
  [k=1] (14.9s, 1842 chars): Based on the given context, there is no specific mention of the correct spark plug gap for a Model T Ford. However, it does state that "All wire connections to spark plugs, coil box and commutator sho...

  Q: How do I fix a slipping transmission band?
  [k=1] (7.2s, 834 chars): To fix a slipping transmission band, loosen the lock nut at the tight side of the transmission cover and turn the adjusting screw (refer to Cut No. 12). Then, remove the transmission cover to access t...

  Q: What oil should I use in a Model T engine?
  [k=1] (8.6s, 1047 chars): The context does not specify which type of oil to use 

In [11]:
# Also examine raw retrieval scores for each k value on one query
PROBE_QUERY = "How do I adjust the carburetor on a Model T?"
print(f"Retrieval scores for: '{PROBE_QUERY}'\n")
for k in K_VALUES:
    results = retrieve(PROBE_QUERY, top_k=k)
    scores = [s for _, s in results]
    print(f"k={k:2d}: scores = {[f'{s:.3f}' for s in scores]}")
    print(f"       min={min(scores):.3f}, max={max(scores):.3f}, mean={sum(scores)/len(scores):.3f}")


Retrieval scores for: 'How do I adjust the carburetor on a Model T?'

k= 1: scores = ['0.640']
       min=0.640, max=0.640, mean=0.640
k= 3: scores = ['0.640', '0.577', '0.577']
       min=0.577, max=0.640, mean=0.598
k= 5: scores = ['0.640', '0.577', '0.577', '0.567', '0.567']
       min=0.567, max=0.640, mean=0.586
k=10: scores = ['0.640', '0.577', '0.577', '0.567', '0.567', '0.558', '0.558', '0.538', '0.534', '0.534']
       min=0.534, max=0.640, mean=0.565
k=20: scores = ['0.640', '0.577', '0.577', '0.567', '0.567', '0.558', '0.558', '0.538', '0.534', '0.534', '0.533', '0.533', '0.531', '0.531', '0.529', '0.526', '0.526', '0.525', '0.525', '0.515']
       min=0.515, max=0.640, mean=0.546


## Analysis — Results from ModelTService corpus (1,286 chunks), Colab T4

> First run with corpus-specific index (`modelTService_512_128` cache loaded ✓).
> No cross-corpus contamination — all 16 documents are Model T manual files.

### Per-Query Quality by k

| Query | k=1 | k=3 | k=5 | k=10 | k=20 |
|-------|-----|-----|-----|------|------|
| Carburetor | ⚠️ Over-hedged, very long (2653 chars, 24.8s) | ⚠️ Still vague, suggests "refer to manual" | ✅ Quotes "For the convenience of the driver", begins to be useful | ✅ Best — numbered steps, locates adjusting rod on dashboard | ✅ Good — cites Answer Nos. 43/45 from Q&A section, comprehensive |
| Spark plug gap | ✅ Honest: "no specific mention in context" | ⚠️ Says ~7/8" (OCR-garbled text: "sparking points sho…") | ⚠️ Says 7/8" with OCR quote "should be 74" (garbled) | ⚠️ Says 7/4" (OCR artifact) | ⚠️ Says 7/16" (another OCR variant) |
| Transmission band | ✅ Correct at k=1: lock nut, adjusting screw, Cut No. 12 | ✅ Adds slow-speed band detail | ✅ Adds brake and reverse bands | ✅ Most concise, actionable | ✅ Adds directional: "turn screw to the right" |
| Engine oil | ⚠️ No type — only "level midway between two cocks" | ✅ Found Model T manual oil section: pour off, refill (OCR garbled but correct source) | ⚠️ Regressed: "all parts properly oiled at factory" — missed oil type chunk | ✅ Best: "Mobil Oil C or Whittemore's Wort Gear Brotectit", upper oil plug level | ✅ Same as k=10, slightly different OCR spelling |
| Engine timing | ✅ Consistent: Bendix drive shaft, loosen shaft cover screws | ✅ Same Bendix content | ✅ Same Bendix content | ✅ Same + more detail on alignment | ❌ Regressed: describes modern timing belt tensioner and camshaft marks — wrong engine entirely |

### Latency and Output Length

| k | Carburetor latency | Spark plug latency | Notes |
|---|--------------------|--------------------|-------|
| 1 | 24.8s (2653 chars) | 14.9s (1842 chars) | Long output at k=1 — model over-explains when context is thin |
| 3 | 8.4s (1076 chars) | 10.0s (1104 chars) | Latency drops sharply — better context → shorter, more direct answers |
| 5 | 13.5s (1615 chars) | 4.2s (443 chars) | Varies by query — output length drives latency more than context size |
| 10 | 9.9s (1152 chars) | 4.4s (366 chars) | Shorter, more confident answers at k=10 |
| 20 | 13.0s (1259 chars) | 5.6s (512 chars) | Slight increase from irrelevant context at k=20 |

### Retrieval Score Distribution (Carburetor probe query)

| k | Score range | Gap rank1→rank2 | Observation |
|---|-------------|-----------------|-------------|
| 1 | 0.640 | — | Single best chunk |
| 3 | 0.577–0.640 | **0.063** | Clear winner at rank 1; ranks 2-3 also relevant |
| 5 | 0.567–0.640 | 0.063 | All 5 chunks score >0.56 — all relevant |
| 10 | 0.534–0.640 | 0.063 | Scores still >0.53 — borderline useful |
| 20 | 0.515–0.640 | 0.063 | Chunks 15–20 score ~0.52 — likely marginal relevance |

---

### Questions Answered

**At what k does adding more context stop helping?**
**k=5 is the sweet spot** for this corpus and chunk size (512 chars). Most queries are well-answered at k=5 and there is little gain beyond it. k=10 is occasionally better (carburetor, oil) but also produces longer and sometimes noisier outputs. The score data supports this: ranks 1–5 score 0.567–0.640 (all clearly relevant), while ranks 6–10 drop to 0.534–0.558 (marginal) and ranks 11–20 drop below 0.533.

**When does too much context hurt?**
The engine timing query at **k=20 is the clearest failure**: a modern timing belt description (camshaft marks, tensioner bolt) crept into the context and the model described a procedure completely wrong for a Model T magneto ignition. The lower-scoring chunks at k=20 (~0.515–0.529) matched "engine timing" semantically but came from unrelated text. This is a direct demonstration that irrelevant high-rank context can corrupt otherwise correct answers.

The oil query also shows non-monotonic quality: k=3 finds the right section, k=5 regresses to "all parts oiled at factory" (a less relevant chunk displaced the good one in the 5-chunk window), then k=10 recovers with "Mobil Oil C." Adding more chunks doesn't guarantee better answers — chunk ranking order matters.

**How does k interact with chunk size (512 chars)?**
At 512-char chunks:
- k=5 → ~2,560 chars of context → fits cleanly, model stays focused
- k=10 → ~5,120 chars → model handles it but answers get shorter/more selective
- k=20 → ~10,240 chars → model starts missing the best chunks and synthesizing from marginal ones

A smaller chunk size (e.g., 256 chars) would require a higher k to accumulate the same information; a larger chunk size (e.g., 1024 chars) would need a lower k before hitting the same context limit. k and chunk_size are inversely proportional for achieving the same effective context coverage.

**Notable improvement vs. Ex 1–3:**
The ModelTService-only corpus **fixed the engine oil query** — at k=10 it retrieved "Mobil Oil C" from the actual Model T manual, whereas the combined index in Ex 1–3 always retrieved the Learjet engine manual instead. Corpus scoping is more impactful than k tuning.


In [12]:
import pickle

def save_index(path):
    faiss.write_index(index, f"{path}.faiss")
    with open(f"{path}.chunks", 'wb') as f:
        pickle.dump(all_chunks, f)
    print(f"✓ Saved {path}.faiss + .chunks")

save_index("rag_index")


✓ Saved rag_index.faiss + .chunks
